In [ ]:
# Step 1: Load KWIC and NMF outputs

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

KWIC_FILE = OUTPUT_DIR / "institution_kwic_results_v2.csv"
NMF_FILE = OUTPUT_DIR / "step10_nmf_document_topic_weights.csv"

kwic = pd.read_csv(KWIC_FILE)
nmf = pd.read_csv(NMF_FILE)

print("KWIC shape:", kwic.shape)
print("NMF shape:", nmf.shape)

print("\nKWIC columns:")
print(kwic.columns.tolist())

print("\nNMF columns:")
print(nmf.columns.tolist())

In [ ]:
# Step 2: Restrict to substantive categories and check NMF mapping

SUBSTANTIVE_CATEGORIES = [
    "police",
    "legal",
    "support_sector"
]

# Keep only the three categories selected for substantive annotation
kwic_sub = kwic[
    kwic["category"].isin(SUBSTANTIVE_CATEGORIES)
].copy()

nmf_sub = nmf[
    nmf["category"].isin(SUBSTANTIVE_CATEGORIES)
].copy()

print("Substantive KWIC records:", len(kwic_sub))
print("Substantive NMF documents:", len(nmf_sub))

print("\nKWIC records by category:")
print(kwic_sub["category"].value_counts())

print("\nUnique transcripts by KWIC category:")
print(
    kwic_sub.groupby("category")["transcript_id"]
    .nunique()
    .sort_index()
)

print("\nNMF documents by category:")
print(nmf_sub["category"].value_counts())

# Check whether transcript_id + category uniquely identifies each NMF document
nmf_duplicates = nmf_sub.duplicated(
    subset=["transcript_id", "category"],
    keep=False
)

print(
    "\nDuplicate transcript_id + category rows in NMF:",
    nmf_duplicates.sum()
)

In [ ]:
# Step 3: Map NMF topic information to KWIC records

topic_cols = [
    f"topic_{i}" for i in range(1, 9)
] + [
    "dominant_topic",
    "dominant_topic_weight"
]

kwic_nmf = kwic_sub.merge(
    nmf_sub[
        ["transcript_id", "category"] + topic_cols
    ],
    on=["transcript_id", "category"],
    how="left",
    validate="many_to_one"
)

print("Merged shape:", kwic_nmf.shape)

print(
    "Rows missing NMF information:",
    kwic_nmf["dominant_topic"].isna().sum()
)

print("\nDominant-topic distribution by category:")
print(
    pd.crosstab(
        kwic_nmf["category"],
        kwic_nmf["dominant_topic"]
    )
)

In [ ]:
# Step 4: Inspect document-level topic structure for substantive categories

doc_topic_counts = pd.crosstab(
    nmf_sub["category"],
    nmf_sub["dominant_topic"]
)

print("Document-level dominant-topic counts:")
print(doc_topic_counts)

print("\nDocument-level dominant-topic proportions:")
doc_topic_props = pd.crosstab(
    nmf_sub["category"],
    nmf_sub["dominant_topic"],
    normalize="index"
).round(3)

print(doc_topic_props)

print("\nMean topic weights by category:")
mean_topic_weights = (
    nmf_sub
    .groupby("category")[
        [f"topic_{i}" for i in range(1, 9)]
    ]
    .mean()
    .round(3)
)

print(mean_topic_weights)

In [ ]:
# Step 5: Inspect KWIC density per transcript-category document

kwic_density = (
    kwic_nmf
    .groupby(
        ["transcript_id", "category", "dominant_topic"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "n_kwic_hits"})
)

print("KWIC hits per transcript-category document:")
print(
    kwic_density["n_kwic_hits"]
    .describe()
    .round(2)
)

print("\nKWIC density by category:")
print(
    kwic_density
    .groupby("category")["n_kwic_hits"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nHighest-hit documents:")
print(
    kwic_density
    .sort_values("n_kwic_hits", ascending=False)
    .head(15)
    .to_string(index=False)
)

In [ ]:
# Step 6: Inspect keyword distribution within substantive categories

keyword_counts = (
    kwic_nmf
    .groupby(["category", "keyword"])
    .size()
    .reset_index(name="n_kwic_hits")
)

keyword_counts["category_total"] = (
    keyword_counts
    .groupby("category")["n_kwic_hits"]
    .transform("sum")
)

keyword_counts["proportion"] = (
    keyword_counts["n_kwic_hits"]
    / keyword_counts["category_total"]
)

for category in SUBSTANTIVE_CATEGORIES:
    print(f"\n--- {category.upper()} ---")
    
    display(
        keyword_counts[
            keyword_counts["category"] == category
        ]
        .sort_values("n_kwic_hits", ascending=False)
        [["keyword", "n_kwic_hits", "proportion"]]
        .head(15)
        .reset_index(drop=True)
    )

## Sampling rationale before formal allocation

The substantive annotation pool contains 6,852 KWIC records across three institutional retrieval categories:

- police: 3,368 KWIC records across 139 transcripts
- legal: 3,098 KWIC records across 133 transcripts
- support_sector: 386 KWIC records across 93 transcripts

The NMF representation contains one document for each transcript × category combination, giving 365 substantive documents in total. The mapping between NMF documents and KWIC records is complete and one-to-many: each NMF document can correspond to multiple KWIC hits.

Inspection of KWIC density shows substantial imbalance across documents. The median number of KWIC hits per transcript-category document is 13, but some documents contain more than 100 hits. Legal and police documents are especially dense, while support-sector documents contain far fewer hits on average. Direct random sampling from the KWIC pool would therefore over-represent high-hit transcripts.

Keyword frequencies are also highly uneven. The police category is dominated by the keyword `police`, while legal is strongly concentrated around `court`. Support-sector keywords are more distributed across terms such as `refuge`, `charity`, `support worker`, and `victim support`. Pure keyword-proportional sampling would therefore reduce lexical diversity, particularly for police and legal cases.

Document-level NMF results show that:

- police-category documents are mainly concentrated in Topics 1 and 5, with smaller contributions from Topics 8 and 6;
- legal-category documents are mainly concentrated in Topics 6, 5 and 8;
- support-sector documents are mainly concentrated in Topics 4 and 5, with a smaller but notable Topic 7 component.

Based on these patterns, the formal annotation sample used a two-stage design:

1. selected transcript × category documents using category-stratified, NMF-informed purposive sampling;
2. selected one focal ±50-word KWIC window from each selected document for manual annotation.

This design preserved the original KWIC annotation unit while reducing bias from transcript-level hit density and improving coverage of major and contrasting narrative patterns.

In [ ]:
# Step 7: Define provisional topic allocation for annotation sampling

topic_allocation = {
    "police": {
        1: 13,
        5: 11,
        8: 3,
        6: 2,
        7: 1
    },
    "legal": {
        6: 14,
        5: 8,
        8: 7,
        1: 1
    },
    "support_sector": {
        4: 13,
        5: 8,
        7: 4,
        1: 2,
        2: 1,
        8: 1,
        6: 1
    }
}

for category, allocation in topic_allocation.items():
    total = sum(allocation.values())
    print(f"{category}: {total}")
    print(allocation)

In [ ]:
# Step 8: Validate topic allocation against available NMF documents

allocation_check = []

for category, allocation in topic_allocation.items():
    for topic, target_n in allocation.items():
        
        available_n = len(
            nmf_sub[
                (nmf_sub["category"] == category) &
                (nmf_sub["dominant_topic"] == topic)
            ]
        )
        
        allocation_check.append({
            "category": category,
            "dominant_topic": topic,
            "available_documents": available_n,
            "target_sample": target_n,
            "sufficient": available_n >= target_n
        })

allocation_check = pd.DataFrame(allocation_check)

display(allocation_check)

print(
    "\nAll allocation targets feasible:",
    allocation_check["sufficient"].all()
)

print(
    "Total planned sample:",
    allocation_check["target_sample"].sum()
)

In [ ]:
# Step 9: Inspect transcript overlap across substantive categories

transcript_category_presence = (
    nmf_sub
    .groupby("transcript_id")["category"]
    .agg(list)
    .reset_index()
)

transcript_category_presence["n_categories"] = (
    transcript_category_presence["category"].apply(len)
)

print("Number of substantive categories represented per transcript:")
print(
    transcript_category_presence["n_categories"]
    .value_counts()
    .sort_index()
)

category_sets = {
    category: set(
        nmf_sub.loc[
            nmf_sub["category"] == category,
            "transcript_id"
        ]
    )
    for category in SUBSTANTIVE_CATEGORIES
}

print("\nPairwise transcript overlap:")

for i, cat1 in enumerate(SUBSTANTIVE_CATEGORIES):
    for cat2 in SUBSTANTIVE_CATEGORIES[i + 1:]:
        overlap = category_sets[cat1] & category_sets[cat2]
        print(f"{cat1} × {cat2}: {len(overlap)} transcripts")

all_three = (
    category_sets["police"]
    & category_sets["legal"]
    & category_sets["support_sector"]
)

print(
    "\nTranscripts represented in all three categories:",
    len(all_three)
)

In [ ]:
# Step 10: Sample transcript-category documents within topic strata

RANDOM_SEED = 42

selected_docs = []

for category, allocation in topic_allocation.items():
    for topic, target_n in allocation.items():

        stratum = nmf_sub[
            (nmf_sub["category"] == category) &
            (nmf_sub["dominant_topic"] == topic)
        ].copy()

        sampled = stratum.sample(
            n=target_n,
            random_state=RANDOM_SEED + topic
        )

        selected_docs.append(sampled)

selected_docs = pd.concat(
    selected_docs,
    ignore_index=True
)

print("Selected transcript-category documents:", len(selected_docs))

print("\nSelected documents by category:")
print(
    selected_docs["category"]
    .value_counts()
)

print("\nSelected documents by category and dominant topic:")
print(
    pd.crosstab(
        selected_docs["category"],
        selected_docs["dominant_topic"]
    )
)

print(
    "\nUnique transcripts represented:",
    selected_docs["transcript_id"].nunique()
)

print(
    "Repeated transcripts across categories:",
    selected_docs["transcript_id"].duplicated().sum()
)

In [ ]:
# Step 11: Build KWIC candidate pool for selected transcript-category documents

selected_keys = selected_docs[
    ["transcript_id", "category"]
].drop_duplicates()

candidate_kwic = kwic_nmf.merge(
    selected_keys,
    on=["transcript_id", "category"],
    how="inner",
    validate="many_to_one"
)

print("Selected documents:", len(selected_keys))
print("Candidate KWIC records:", len(candidate_kwic))

print("\nCandidate KWIC records by category:")
print(
    candidate_kwic["category"]
    .value_counts()
)

print("\nCandidate KWIC hits per selected document:")
candidate_density = (
    candidate_kwic
    .groupby(["transcript_id", "category"])
    .size()
)

print(
    candidate_density
    .describe()
    .round(2)
)

print(
    "\nAll 90 selected documents have at least one KWIC:",
    candidate_density.size == 90
)

In [ ]:
# Step 12: Inspect keyword availability within selected documents

doc_keyword_summary = (
    candidate_kwic
    .groupby(["transcript_id", "category"])
    .agg(
        n_kwic_hits=("keyword", "size"),
        n_unique_keywords=("keyword", "nunique"),
        keywords=("keyword", lambda x: sorted(set(x)))
    )
    .reset_index()
)

print("Unique keywords available per selected document:")
print(
    doc_keyword_summary["n_unique_keywords"]
    .describe()
    .round(2)
)

print("\nDocuments with only one available keyword:")
print(
    (doc_keyword_summary["n_unique_keywords"] == 1).sum()
)

print("\nDocuments with multiple available keywords:")
print(
    (doc_keyword_summary["n_unique_keywords"] > 1).sum()
)

print("\nKeyword availability by category:")
print(
    doc_keyword_summary
    .groupby("category")["n_unique_keywords"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nExamples of keyword sets:")
display(
    doc_keyword_summary
    .sort_values(
        ["n_unique_keywords", "n_kwic_hits"],
        ascending=False
    )
    .head(15)
)

In [ ]:
# Step 13: Inspect nearby and potentially overlapping KWIC hits

candidate_sorted = candidate_kwic.sort_values(
    ["transcript_id", "category", "match_start"]
).copy()

candidate_sorted["previous_match_start"] = (
    candidate_sorted
    .groupby(["transcript_id", "category"])["match_start"]
    .shift(1)
)

candidate_sorted["distance_from_previous"] = (
    candidate_sorted["match_start"]
    - candidate_sorted["previous_match_start"]
)

# Because the annotation window is ±50 words but match positions are
# character-based, this is only a diagnostic for nearby hits,
# not an exact measure of KWIC-window overlap.
NEARBY_CHAR_THRESHOLD = 300

candidate_sorted["near_previous_hit"] = (
    candidate_sorted["distance_from_previous"]
    <= NEARBY_CHAR_THRESHOLD
)

print("Candidate KWIC records:", len(candidate_sorted))

print(
    "\nHits occurring within",
    NEARBY_CHAR_THRESHOLD,
    "characters of the previous hit:",
    candidate_sorted["near_previous_hit"].sum()
)

print(
    "Proportion:",
    round(candidate_sorted["near_previous_hit"].mean(), 3)
)

print("\nNearby-hit proportion by category:")
print(
    candidate_sorted
    .groupby("category")["near_previous_hit"]
    .mean()
    .round(3)
)

print("\nExamples of nearby hits:")
display(
    candidate_sorted.loc[
        candidate_sorted["near_previous_hit"],
        [
            "transcript_id",
            "category",
            "keyword",
            "matched_text",
            "match_start",
            "distance_from_previous",
            "context"
        ]
    ].head(15)
)

In [ ]:
# Step 14: Select one focal KWIC per selected transcript-category document

RANDOM_SEED = 42

final_sample = (
    candidate_kwic
    .groupby(["transcript_id", "category"], group_keys=False)
    .sample(n=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

print("Final annotation sample size:", len(final_sample))

print("\nFinal sample by category:")
print(
    final_sample["category"]
    .value_counts()
)

print("\nUnique transcripts represented:")
print(
    final_sample["transcript_id"].nunique()
)

print("\nKeyword distribution in final sample:")
print(
    pd.crosstab(
        final_sample["category"],
        final_sample["keyword"]
    )
)

In [ ]:
# Step 15: Run final sample quality checks

print("=== FINAL SAMPLE QUALITY CHECK ===")

# 1. Sample size
print("\nSample size:", len(final_sample))

# 2. Category balance
print("\nCategory counts:")
print(final_sample["category"].value_counts())

# 3. Topic allocation
print("\nCategory × dominant topic:")
print(
    pd.crosstab(
        final_sample["category"],
        final_sample["dominant_topic"]
    )
)

# 4. Unique transcript-category combinations
print(
    "\nUnique transcript-category combinations:",
    final_sample[
        ["transcript_id", "category"]
    ].drop_duplicates().shape[0]
)

# 5. Unique transcripts
print(
    "Unique transcripts:",
    final_sample["transcript_id"].nunique()
)

# 6. Missing values in essential fields
essential_cols = [
    "transcript_id",
    "category",
    "keyword",
    "context",
    "dominant_topic",
    "dominant_topic_weight"
]

print("\nMissing values in essential fields:")
print(
    final_sample[essential_cols]
    .isna()
    .sum()
)

# 7. Duplicate KWIC locations
duplicate_locations = final_sample.duplicated(
    subset=[
        "transcript_id",
        "category",
        "match_start",
        "match_end"
    ]
).sum()

print(
    "\nDuplicate KWIC locations:",
    duplicate_locations
)

# 8. Overall pass/fail checks
checks = {
    "sample_size_90": len(final_sample) == 90,
    "30_per_category": (
        final_sample["category"]
        .value_counts()
        .eq(30)
        .all()
    ),
    "90_unique_document_units": (
        final_sample[
            ["transcript_id", "category"]
        ]
        .drop_duplicates()
        .shape[0] == 90
    ),
    "no_missing_essential_fields": (
        final_sample[essential_cols]
        .isna()
        .sum()
        .sum() == 0
    ),
    "no_duplicate_kwic_locations": duplicate_locations == 0
}

print("\nFinal checks:")
for check, passed in checks.items():
    print(f"{check}: {passed}")

print(
    "\nALL CHECKS PASSED:",
    all(checks.values())
)

## Formal annotation framework

The final manual annotation sample consists of 90 KWIC passages, with 30 passages drawn from each of the three substantive institutional categories: police, legal, and support sector. These passages were selected through the category-stratified and NMF-informed sampling procedure described above.

The annotation unit remains the keyword-centred ±50-word KWIC passage. NMF topics are used only to support sample construction and pattern coverage; they are not treated as manual annotation labels and do not determine the interpretation of individual passages.

The formal annotation framework uses the four dimensions specified in the annotation codebook:

1. **valid_mention**  
   Records whether the passage contains a substantively interpretable institutional encounter relevant to the research question.

2. **institutional_role**  
   Records the role or action attributed to the institution within the survivor's account, using the previously established institutional-role labels.

3. **justice_dimension**  
   Records the dimension of justice or injustice expressed in the passage, using the previously established coding scheme informed by the Kaleidoscopic Justice framework.

4. **evaluative_direction**  
   Records the direction of the survivor's evaluation of the institutional encounter, using the previously established evaluative labels.

The definitions and permitted labels for these four dimensions remain unchanged from the established annotation codebook. This preserves consistency between the earlier annotation work and the final substantive annotation stage.

Two additional free-text fields are retained for annotation management:

- **ambiguity_note**: brief documentation of uncertain or borderline coding decisions;
- **researcher_note**: brief interpretive notes that may support subsequent close reading and qualitative interpretation.

Computational metadata, including transcript ID, institutional category, retrieval keyword, NMF dominant topic, and topic weights, are retained alongside the annotation fields but are not manually coded.

The purpose of manual annotation is not to train a predictive classifier. Instead, it provides a structured interpretive layer through which computationally identified patterns can be validated, compared, and examined in relation to survivors' descriptions of institutional justice and injustice.

In [ ]:
# Step 16: Create the formal annotation sheet

annotation_sheet = final_sample.copy()

# Add a stable annotation ID
annotation_sheet.insert(
    0,
    "annotation_id",
    [f"ANN_{i:03d}" for i in range(1, len(annotation_sheet) + 1)]
)

# Add manual annotation fields
annotation_sheet["valid_mention"] = ""
annotation_sheet["institutional_role"] = ""
annotation_sheet["justice_dimension"] = ""
annotation_sheet["evaluative_direction"] = ""
annotation_sheet["ambiguity_note"] = ""
annotation_sheet["researcher_note"] = ""

# Select and order columns for annotation
annotation_columns = [
    # Identification
    "annotation_id",
    "transcript_id",
    "category",

    # KWIC information
    "keyword",
    "matched_text",
    "context",

    # Computational sampling information
    "dominant_topic",
    "dominant_topic_weight",

    # Manual annotation fields
    "valid_mention",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction",
    "ambiguity_note",
    "researcher_note"
]

annotation_sheet = annotation_sheet[annotation_columns]

# Export
ANNOTATION_FILE = OUTPUT_DIR / "step16_formal_annotation_sample.csv"

annotation_sheet.to_csv(
    ANNOTATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Annotation sheet shape:", annotation_sheet.shape)
print("Saved to:", ANNOTATION_FILE)

display(annotation_sheet.head())